# Introducción

En este proyecto se va a analizar un dataset con estadísticas de 1537 rookies de la NBA, es decir, jugadores en su primer año de carrera. Para ello se va a utilizar datos sacados de Kaggle, concretamente https://www.kaggle.com/datasets/thedevastator/nba-rookies-performance-statistics-and-minutes-p. El objetivo principal de este trabajo es visualizar estos datos mediante gráficas y extraer conclusiones. Ademas, se realizará un modelo predictivo para predecir si un jugador entrará en el Hall of Fame o no basándose puramente en sus estadísticas de año rookie

In [ ]:
!pip install pandas numpy dash dash-bootstrap-components plotly scikit-learn

In [ ]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

from dash import Dash, html, dcc, Input, Output, State
import dash_bootstrap_components as dbc
from jupyter_dash import JupyterDash as Dash
from google.colab import output

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import warnings

# Analisis de datos (EDA)

In [ ]:
data= pd.read_csv("/content/drive/MyDrive/TECNUN 2025-26/VisDatos_2cuatri/Dash Project/NBA Rookies by Year_Hall of Fame Class.csv")

In [ ]:
data.head()

,index,Name,Hall of Fame Class,Year Drafted,GP,MIN,PTS,FGM,FGA,FG%,...,FTA,FT%,OREB,DREB,REB,AST,STL,BLK,TOV,EFF
0,0,Jeff Taylor,NaN,1982.0,44.0,17.6,3.6,1.5,3.6,40.0,...,1.0,65.2,0.6,1.2,1.8,2.5,0.9,0.3,1.4,5.2
1,1,Charles Smith,NaN,1988.0,71.0,30.4,16.3,6.1,12.4,49.5,...,5.5,72.5,2.4,4.1,6.5,1.5,1.0,1.3,2.1,16.7
2,2,Mark Davis,NaN,1988.0,33.0,7.8,3.8,1.5,3.1,48.0,...,1.0,82.4,0.5,0.6,1.1,0.4,0.4,0.1,0.4,3.8
3,3,Charles Smith,NaN,1989.0,60.0,8.7,2.9,1.0,2.2,44.4,...,1.3,69.7,0.2,0.9,1.2,1.7,0.6,0.1,0.6,4.1
4,4,Michael Smith,NaN,1989.0,65.0,9.5,5.0,2.1,4.4,47.6,...,1.0,82.8,0.6,0.9,1.5,1.2,0.1,0.0,0.8,4.6


In [ ]:
data.describe()

,index,Hall of Fame Class,Year Drafted,GP,MIN,PTS,FGM,FGA,FG%,3P Made,...,FTA,FT%,OREB,DREB,REB,AST,STL,BLK,TOV,EFF
count,2013.000000,26.000000,1537.000000,1537.000000,1537.000000,1537.000000,1537.000000,1537.000000,1537.000000,1537.000000,...,1537.000000,1537.000000,1537.000000,1537.000000,1537.000000,1537.000000,1537.000000,1537.000000,1537.000000,1537.000000
mean,1006.000000,2009.538462,1997.847105,60.148341,17.539297,6.708523,2.587248,5.791737,44.170332,0.259792,...,1.793364,70.363045,1.014379,2.057840,3.071763,1.495446,0.602212,0.389070,1.179440,7.362199
std,581.247366,4.666740,10.774472,17.478077,8.121132,4.264164,1.643950,3.494965,6.147384,0.394456,...,1.307633,10.729865,0.774627,1.357413,2.049300,1.424732,0.401197,0.466966,0.709033,4.553934
min,0.000000,1999.000000,1980.000000,11.000000,3.100000,0.700000,0.300000,0.800000,20.500000,0.000000,...,0.000000,0.000000,0.000000,0.100000,0.200000,0.000000,0.000000,0.000000,0.100000,0.200000
25%,503.000000,2006.500000,1989.000000,47.000000,11.100000,3.700000,1.400000,3.300000,40.300000,0.000000,...,0.900000,64.500000,0.400000,1.100000,1.600000,0.500000,0.300000,0.100000,0.700000,3.900000
50%,1006.000000,2010.000000,1998.000000,62.000000,16.100000,5.500000,2.100000,4.800000,44.000000,0.100000,...,1.400000,71.400000,0.800000,1.800000,2.600000,1.000000,0.500000,0.200000,1.000000,6.300000
75%,1509.000000,2012.750000,2007.000000,77.000000,22.800000,8.700000,3.300000,7.400000,47.900000,0.400000,...,2.300000,77.600000,1.400000,2.700000,4.000000,1.900000,0.800000,0.500000,1.500000,9.700000
max,2012.000000,2016.000000,2016.000000,82.000000,40.900000,28.200000,10.200000,19.800000,73.700000,2.300000,...,10.200000,100.000000,5.400000,9.600000,13.900000,10.600000,2.500000,5.000000,4.400000,30.600000


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2013 entries, 0 to 2012
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   index               2013 non-null   int64  
 1   Name                1537 non-null   object 
 2   Hall of Fame Class  26 non-null     float64
 3   Year Drafted        1537 non-null   float64
 4   GP                  1537 non-null   float64
 5   MIN                 1537 non-null   float64
 6   PTS                 1537 non-null   float64
 7   FGM                 1537 non-null   float64
 8   FGA                 1537 non-null   float64
 9   FG%                 1537 non-null   float64
 10  3P Made             1537 non-null   float64
 11  3PA                 1537 non-null   float64
 12  3P%                 1537 non-null   object 
 13  FTM                 1537 non-null   float64
 14  FTA                 1537 non-null   float64
 15  FT%                 1537 non-null   float64
 16  OREB  

In [ ]:
data = data.dropna(subset=['Name'])

data['3P%'] = data['3P%'].replace('-', '0')
data['3P%'] = data['3P%'].astype(float)

data['Year Drafted'] = data['Year Drafted'].astype(int)

data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1537 entries, 0 to 1536
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   index               1537 non-null   int64  
 1   Name                1537 non-null   object 
 2   Hall of Fame Class  26 non-null     float64
 3   Year Drafted        1537 non-null   int64  
 4   GP                  1537 non-null   float64
 5   MIN                 1537 non-null   float64
 6   PTS                 1537 non-null   float64
 7   FGM                 1537 non-null   float64
 8   FGA                 1537 non-null   float64
 9   FG%                 1537 non-null   float64
 10  3P Made             1537 non-null   float64
 11  3PA                 1537 non-null   float64
 12  3P%                 1537 non-null   float64
 13  FTM                 1537 non-null   float64
 14  FTA                 1537 non-null   float64
 15  FT%                 1537 non-null   float64
 16  OREB       

In [ ]:
data['Is_HOF'] = data['Hall of Fame Class'].notna().astype(int)

# Rellenamos los nulos originales con 'No' para tenerlo limpio
data['Hall of Fame Class'] = data['Hall of Fame Class'].fillna('No')

print("Jugadores en el Hall of Fame (1):", data['Is_HOF'].sum())
print("Resto de jugadores (0):", (data['Is_HOF'] == 0).sum())

Jugadores en el Hall of Fame (1): 26
Resto de jugadores (0): 1511


# Modelo de predicción
Entrenamos el modelo Logistic Regression para despues utilizarlo en el dash

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression

# Filtramos: Solo usamos para entrenar a jugadores drafteados hasta 2002.
# Así evitamos que LeBron (2003, activo) confundan al modelo con un "0 falso".
data_entrenamiento = data[data['Year Drafted'] <= 1992]
print(data_entrenamiento.shape[0])#Porcentaje de datos entrenamiento

columnas_modelo = ['MIN', 'PTS', 'REB', 'AST']
X = data_entrenamiento[columnas_modelo]
y = data_entrenamiento['Is_HOF']

# Entrenamos el modelo con los datos filtrados
modelo_lr = LogisticRegression(class_weight='balanced', random_state=42)
modelo_lr.fit(X, y)
# 3. Probamos con nuestro jugador inventado
jugador_inventado = pd.DataFrame([[20.0, 13.0, 4.0, 6.0]], columns=columnas_modelo)
probabilidad = modelo_lr.predict_proba(jugador_inventado)[0][1]

print(f"Probabilidad de que este jugador inventado sea Hall of Fame: {probabilidad * 100:.1f}%")

569
Probabilidad de que este jugador inventado sea Hall of Fame: 93.9%


# Aplicación Dash

In [ ]:
fuente_google = "https://fonts.googleapis.com/css2?family=Montserrat:wght@400;600;800&display=swap"
app = Dash(__name__, external_stylesheets=[dbc.themes.FLATLY, fuente_google])

nombres_jugadores = data['Name'].unique().tolist()

# Nos aseguramos de que los nombres de jugadores sean validos:
jugador1_default = nombres_jugadores[0] if len(nombres_jugadores) > 0 else None
jugador2_default = nombres_jugadores[1] if len(nombres_jugadores) > 1 else None



# --- DISEÑO DEL TÍTULO ---
custom_header = html.Div([
    html.Div([
        html.H1("Análisis de Rookies NBA",
                style={'fontFamily': 'Montserrat, sans-serif', 'fontWeight': '800', 'color': '#2c3e50'},
                className="mb-2"),
        html.Img(
            src="https://upload.wikimedia.org/wikipedia/en/0/03/National_Basketball_Association_logo.svg",
            style={'height': '60px', 'marginLeft': '15px'}
        )
    ], style={'display': 'flex', 'alignItems': 'center'}),

    html.P("Explorando el rendimiento de los novatos y su camino al Hall of Fame",
           style={'fontFamily': 'Montserrat, sans-serif', 'fontWeight': '400', 'color': '#5a6b7c', 'fontSize': '1.2rem'},
           className="mb-0"),
    ], style={
        'backgroundColor': '#eef2f5',
        'padding': '35px',
        'borderRadius': '12px',
        'marginBottom': '30px',
        'boxShadow': '0 4px 10px rgba(0,0,0,0.05)'
    })


tab1_content = dbc.Card(dbc.CardBody([

    dbc.Row([

# --- GRÁFICO 1: EVO: TIPOS DE TIRO ---

        dbc.Col([

            html.H5("Evolución de cada Tipo de Tiro", style={'fontFamily': 'Montserrat', 'color': '#2c3e50'}, className="fw-bold text-center mb-3"),

            # El Dropdown
            dcc.Dropdown(
                id='dropdown-tiros',
                options=[
                    {'label': 'Triples Intentados (3PA)', 'value': '3PA'},
                    {'label': 'Tiros de Campo Totales (FGA)', 'value': 'FGA'},
                    {'label': 'Tiros Libres Intentados (FTA)', 'value': 'FTA'}
                ],
                value='3PA', # Valor por defecto al cargar la página
                clearable=False, # Evita que el usuario lo deje vacío
                style={'fontFamily': 'Montserrat', 'marginBottom': '10px'}
            ),

            # El espacio real para el gráfico de Plotly
            dcc.Graph(id='grafico-evolucion-tiros', style={'height': '300px'})
                                                                                ], md=6),

# --- GRÁFICO 2: HALL OF FAME ---

        dbc.Col([
            html.H5("¿Qué hace a un Hall of Fame?", style={'fontFamily': 'Montserrat', 'color': '#2c3e50'}, className="fw-bold text-center mb-3"),

            # Nuevo Dropdown para elegir la estadística
            dcc.Dropdown(
                id='dropdown-hof',
                options=[
                    {'label': 'Minutos Jugados (MIN)', 'value': 'MIN'},
                    {'label': 'Puntos (PTS)', 'value': 'PTS'},
                    {'label': 'Rebotes (REB)', 'value': 'REB'},
                    {'label': 'Asistencias (AST)', 'value': 'AST'},
                    {'label': 'Robos (STL)', 'value': 'STL'},
                    {'label': 'Tapones (BLK)', 'value': 'BLK'},
                    {'label': 'Pérdidas (TOV)', 'value': 'TOV'},
                    {'label': 'Eficiencia (EFF)', 'value': 'EFF'},
                    {'label': 'Tiros de Campo Int. (FGA)', 'value': 'FGA'},
                    {'label': 'Tiros Libres Int. (FTA)', 'value': 'FTA'},
                    {'label': 'Triples Intentados (3PA)', 'value': '3PA'},
                ],
                value='PTS',
                clearable=False,
                style={'fontFamily': 'Montserrat', 'marginBottom': '10px'}
            ),
            # Nuevo espacio para el gráfico
            dcc.Graph(id='grafico-hof', style={'height': '300px'})
                                                                                ], md=6),
                                                                                          ], className="mb-4"),

# --- GRÁFICO 3: COMPARADOR ---

    dbc.Row([

        dbc.Col([
            html.H5("Comparador", style={'fontFamily': 'Montserrat', 'color': '#2c3e50'}, className="fw-bold mb-4"),
            html.P("Selecciona dos jugadores para solapar sus perfiles estadísticos y compararlos directamente.", style={'color': '#7f8c8d'}),

            html.Label("🔵 Jugador 1:", style={'fontWeight': 'bold', 'color': '#3498db'}),
            dcc.Dropdown(
                id='dropdown-jugador1',
                options=[{'label': nombre, 'value': nombre} for nombre in nombres_jugadores],
                value="Mark Jackson",
                clearable=False,
                style={'marginBottom': '20px'}
            ),

            html.Label("🟠 Jugador 2:", style={'fontWeight': 'bold', 'color': '#e67e22'}),
            dcc.Dropdown(
                id='dropdown-jugador2',
                options=[{'label': nombre, 'value': nombre} for nombre in nombres_jugadores],
                value="Manute Bol",
                clearable=False
            ),
        ], md=4, style={'backgroundColor': '#f8f9fa', 'padding': '25px', 'borderRadius': '10px'}),

        # Columna Derecha (md=8): El gráfico de radar
        dbc.Col([
            dcc.Graph(id='grafico-radar', style={'height': '400px'})
                                                                                ], md=8)
                                                                                           ], className="align-items-center") # Centra verticalmente los menús con el gráfico
                                                                                                                ]), className="mt-3 border-0 shadow-sm") #cerramos pestaña



tab2_content = dbc.Card(dbc.CardBody([

 # --- GRÁFICO 4: Buscador ---
    html.H4("Buscador de Estrellas", style={'fontFamily': 'Montserrat', 'color': '#2c3e50'}, className="fw-bold text-center mb-2"),
    html.P("Cruza dos estadísticas para ver dónde se agrupan las leyendas. Pasa el ratón por los puntos dorados para ver quiénes son.", className="text-center mb-4", style={'color': '#5a6b7c'}),

    dbc.Row([
        dbc.Col([
            html.Label("Eje Horizontal (X):", style={'fontWeight': 'bold', 'color': '#2c3e50'}),
            dcc.Dropdown(id='scatter-x', options=[{'label': c, 'value': c} for c in ['MIN', 'PTS', 'REB', 'AST', 'TOV', 'EFF']], value='MIN', clearable=False)
        ], md=6),
        dbc.Col([
            html.Label("Eje Vertical (Y):", style={'fontWeight': 'bold', 'color': '#2c3e50'}),
            dcc.Dropdown(id='scatter-y', options=[{'label': c, 'value': c} for c in ['MIN', 'PTS', 'REB', 'AST', 'TOV', 'EFF']], value='PTS', clearable=False)
        ], md=6),
    ], className="mb-3"),

    dcc.Graph(id='grafico-scatter', style={'height': '400px'}),

    html.Hr(style={'borderColor': '#eef2f5', 'borderWidth': '2px', 'margin': '40px 0'}),
    html.H4("Predictor de Hall of Fame", style={'fontFamily': 'Montserrat', 'color': '#2c3e50'}, className="fw-bold text-center mb-3"),
    html.P("Introduce las estadísticas proyectadas para la temporada de rookie del jugador. El modelo de Regresión logística calculará su probabilidad de entrar al Hall of Fame basado en datos históricos.", className="text-center mb-5", style={'fontFamily': 'Montserrat', 'color': '#5a6b7c'}),

    #inputs
    dbc.Row([
        dbc.Col([html.Label("Minutos (MIN):", style={'fontFamily': 'Montserrat', 'fontWeight': '600'}), dbc.Input(id="input-min", type="number", value=15.0, step=0.1)], md=2),
        dbc.Col([html.Label("Puntos (PTS):", style={'fontFamily': 'Montserrat', 'fontWeight': '600'}), dbc.Input(id="input-pts", type="number", value=8.0, step=0.1)], md=2),
        dbc.Col([html.Label("Rebotes (REB):", style={'fontFamily': 'Montserrat', 'fontWeight': '600'}), dbc.Input(id="input-reb", type="number", value=3.0, step=0.1)], md=2),
        dbc.Col([html.Label("Asist. (AST):", style={'fontFamily': 'Montserrat', 'fontWeight': '600'}), dbc.Input(id="input-ast", type="number", value=2.0, step=0.1)], md=2),

        # Botón
        dbc.Col([html.Br(), dbc.Button("Predecir Futuro", id="btn-predecir", color="primary", className="w-100", style={'fontFamily': 'Montserrat', 'fontWeight': '600'})], md=2),

    ], className="mb-4"),

    # Resultado
    html.Div(id="resultado-prediccion", className="mt-4")

]), className="mt-3 border-0 shadow-sm", style={'padding': '30px'})

# --- LAYOUT ---
app.layout = dbc.Container([
    custom_header,
    dbc.Tabs([
        dbc.Tab(tab1_content, label="Análisis Visual", tab_id="tab-1", label_style={'fontFamily': 'Montserrat', 'fontWeight': '600'}),
        dbc.Tab(tab2_content, label="Modelo de Predicción", tab_id="tab-2", label_style={'fontFamily': 'Montserrat', 'fontWeight': '600'}),
    ], id="tabs", active_tab="tab-1")
], fluid=True, className="p-4")


# Callback del Gráfico 1 (Evo. tipo de tiro)
@app.callback(
    Output('grafico-evolucion-tiros', 'figure'),
    Input('dropdown-tiros', 'value')
)
def actualizar_grafico_tiros(metrica_seleccionada):
    # 1. Agrupamos los datos por año y calculamos la media
    df_agrupado = data.groupby('Year Drafted')[metrica_seleccionada].mean().reset_index()

    # 2.gráfico
    fig = px.line(
        df_agrupado,
        x='Year Drafted',
        y=metrica_seleccionada,
        markers=True, # Ponemos puntitos en cada año
        labels={'Year Drafted': 'Año del Draft', metrica_seleccionada: 'Media por Partido'}
    )

    # 3. Diseño
    fig.update_traces(
        line_color='#2c3e50',
        line_width=3,
        marker=dict(color='#3498db', size=8, line=dict(color='white', width=1.5)) # Azul claro para los puntos
    )

    fig.update_layout(
        margin=dict(l=20, r=20, t=10, b=20),
        plot_bgcolor='rgba(0,0,0,0)', # Fondo transparente
        paper_bgcolor='rgba(0,0,0,0)',
        font_family="Montserrat",
        xaxis=dict(showgrid=True, gridcolor='#eef2f5'), # Cuadrícula muy suave
        yaxis=dict(showgrid=True, gridcolor='#eef2f5')
    )

    return fig

# Callback del Gráfico 2 (Hall of Fame)
@app.callback(
    Output('grafico-hof', 'figure'),
    Input('dropdown-hof', 'value')
)
def actualizar_grafico_hof(metrica):
    data_plot = data.copy()
    data_plot['Leyenda'] = data_plot['Is_HOF'].map({1: 'Salón de la Fama', 0: 'Resto de Jugadores'})

    fig = px.box(
        data_plot,
        x='Leyenda',
        y=metrica,
        color='Leyenda',
        color_discrete_map={'Salón de la Fama': '#2c3e50', 'Resto de Jugadores': '#aab7c4'},
        # Solo dibujamos los puntos que sean valores extremos (outliers)
        points="outliers"
    )

    fig.update_layout(
        margin=dict(l=20, r=20, t=10, b=20), plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
        font_family="Montserrat", xaxis=dict(title='', showgrid=False), yaxis=dict(title='Media por Partido', showgrid=True, gridcolor='#eef2f5'),
        showlegend=False
    )
    return fig


# Callback del Gráfico 3 (Comparador)
@app.callback(
    Output('grafico-radar', 'figure'),
    [Input('dropdown-jugador1', 'value'),
     Input('dropdown-jugador2', 'value')]
)
def actualizar_radar(jugador1, jugador2):
    # Las categorías que vamos a comparar
    categorias = ['PTS', 'REB', 'AST', 'STL', 'BLK']

    fig = go.Figure()

    # Extraemos los datos del Jugador 1 y Jugador 2 (asumimos que existe y cogemos su primer año si hay duplicados)
    datos_j1 = data[data['Name'] == jugador1].iloc[0]
    datos_j2 = data[data['Name'] == jugador2].iloc[0]

    # Preparamos los valores (hay que añadir el primer valor al final para que el polígono se cierre)
    valores_j1 = [datos_j1[cat] for cat in categorias]
    valores_j1.append(valores_j1[0])

    valores_j2 = [datos_j2[cat] for cat in categorias]
    valores_j2.append(valores_j2[0])

    categorias_cerradas = categorias + [categorias[0]]

    #jugador 1
    fig.add_trace(go.Scatterpolar(
        r=valores_j1,
        theta=categorias_cerradas,
        fill='toself', # Esto rellena el área
        name=jugador1,
        line_color='#3498db', # Azul
        fillcolor='rgba(52, 152, 219, 0.4)' # Azul con 40% de opacidad para que se vea el de abajo
    ))

    #jugador 2
    fig.add_trace(go.Scatterpolar(
        r=valores_j2,
        theta=categorias_cerradas,
        fill='toself',
        name=jugador2,
        line_color='#e67e22', # Naranja
        fillcolor='rgba(230, 126, 34, 0.4)' # Naranja con 40% de opacidad
    ))

    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                visible=True,
                showline=False,
                gridcolor='#eef2f5'
            )
        ),
        showlegend=True,
        margin=dict(l=40, r=40, t=20, b=20),
        font_family="Montserrat",
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
    )

    return fig

# Callback del Gráfico 4 (Scatter)

@app.callback(
    Output('grafico-scatter', 'figure'),
    [Input('scatter-x', 'value'), Input('scatter-y', 'value')]
)
def actualizar_scatter(eje_x, eje_y):
    data_plot = data.copy()
    data_plot['Leyenda'] = data_plot['Is_HOF'].map({1: 'Salón de la Fama', 0: 'Resto de Jugadores'})

    # Ordenamos para que los puntos dorados (HOF) se dibujen los últimos y queden por encima de los grises
    data_plot = data_plot.sort_values(by='Is_HOF')

    fig = px.scatter(
        data_plot, x=eje_x, y=eje_y, color='Leyenda',
        hover_name='Name', # Muestra el nombre al pasar el ratón
        color_discrete_map={'Salón de la Fama': '#f1c40f', 'Resto de Jugadores': '#aab7c4'},
        opacity=0.8
    )

    # Hacemos los puntos de los HOF un poquito más grandes visualmente
    fig.update_traces(marker=dict(size=9, line=dict(width=1, color='white')))

    fig.update_layout(
        margin=dict(l=20, r=20, t=10, b=20), plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)', font_family="Montserrat",
        xaxis=dict(showgrid=True, gridcolor='#eef2f5'), yaxis=dict(showgrid=True, gridcolor='#eef2f5'),
        legend=dict(title="", orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    return fig

# Callback del Gráfico 5 (Predicción)
@app.callback(
    Output('resultado-prediccion', 'children'),
    Input('btn-predecir', 'n_clicks'), # El botón actúa como gatillo
    [State('input-min', 'value'),      # Usamos State porque solo queremos leer estos valores
     State('input-pts', 'value'),      # cuando el usuario haga clic en el botón
     State('input-reb', 'value'),
     State('input-ast', 'value'),]
)

def realizar_prediccion(n_clicks, min_val, pts_val, reb_val, ast_val):
    # Si no han hecho clic todavía, no mostramos nada
    if n_clicks is None:
        return html.Div()

    # Asegurarnos de que no hay campos vacíos
    if None in [min_val, pts_val, reb_val, ast_val]:
        return html.Div("Por favor, rellena todos los campos.", className="alert alert-warning text-center")

    # 1. Hacemos la predicción con el modelo entrenado en la celda anterior
    datos_jugador = [[min_val, pts_val, reb_val, ast_val]]
    prob_hof = modelo_lr.predict_proba(datos_jugador)[0][1] * 100

    # 2.recuadro de resultado según la probabilidad
    if prob_hof >= 65:
        estilo = {'backgroundColor': '#fff9e6', 'color': '#856404', 'border': '2px solid #ffeeba', 'padding': '20px', 'borderRadius': '10px'}
        mensaje = "Perfil de HOF ! "
    else:
        estilo = {'backgroundColor': '#eef2f5', 'color': '#383d41', 'border': '2px solid #d6d8db', 'padding': '20px', 'borderRadius': '10px'}
        mensaje = "Jugador de rotación/titular sólido"

    return html.Div([
        html.H3(mensaje, style={'fontFamily': 'Montserrat', 'fontWeight': '800'}, className="text-center mb-2"),
        html.H4(f"Probabilidad de Hall of Fame: {prob_hof:.1f}%", style={'fontFamily': 'Montserrat'}, className="text-center")
    ], style=estilo)

# --- EJECUCIÓN CON GOOGLE COLAB ---
if __name__ == '__main__':
    from google.colab import output
    print("Lanzando la aplicación...")

    output.serve_kernel_port_as_window(8050)

    # Lanzamos Dash
    app.run(port=8050)

Lanzando la aplicación...
Try `serve_kernel_port_as_iframe` instead. 


/usr/local/lib/python3.12/dist-packages/dash/dash.py:644: UserWarning:

JupyterDash is deprecated, use Dash instead.
See https://dash.plotly.com/dash-in-jupyter for more details.



<IPython.core.display.Javascript object>

Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:8050
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [28/Mar/2026 12:23:06] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [28/Mar/2026 12:23:06] "GET /_dash-component-suites/dash/deps/react-dom@18.v4_1_0m1774692306.3.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [28/Mar/2026 12:23:06] "GET /_dash-component-suites/dash_bootstrap_components/_components/dash_bootstrap_components.v2_0_4m1774692308.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [28/Mar/2026 12:23:06] "GET /_dash-component-suites/dash/dcc/dash_core_components.v4_1_0m1774692306.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [28/Mar/2026 12:23:06] "GET /_dash-component-suites/dash/html/dash_html_components.v4_1_0m1774692306.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [28/Mar/2026 12:23:06] "GET /_dash-component-suites/dash/dash_